<a href="https://colab.research.google.com/github/mbilal1267/Machine-Learning-LAB/blob/main/ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 2. Scale pixel values to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# 3. Expand dimension for channels: (28,28) -> (28,28,1)
x_train = np.expand_dims(x_train, axis=-1)  # (60000, 28, 28, 1)
x_test = np.expand_dims(x_test, axis=-1)    # (10000, 28, 28, 1)

# 4. Convert labels to one-hot vectors
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

print(f"x_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}, y_test shape: {y_test.shape}")


x_train shape: (60000, 28, 28, 1), y_train shape: (60000, 10)
x_test shape: (10000, 28, 28, 1), y_test shape: (10000, 10)


In [16]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size=4, hidden_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.hidden_dim = hidden_dim
        self.projection = layers.Dense(hidden_dim)

    def call(self, images):
        # images shape: (batch_size, height, width, channels)
        batch_size = tf.shape(images)[0]
        patch_size = self.patch_size

        # 1. Extract patches
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, patch_size, patch_size, 1],
            strides=[1, patch_size, patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        # patches shape: (batch_size, num_patches_h, num_patches_w, patch_size*patch_size*channels)

        # 2. Flatten patches
        num_patches_h = tf.shape(patches)[1]  # 7
        num_patches_w = tf.shape(patches)[2]  # 7
        patch_dims = tf.shape(patches)[3]     # patch_size*patch_size*channels
        patches = tf.reshape(patches, [batch_size, num_patches_h * num_patches_w, patch_dims])

        # 3. Project patches to hidden_dim
        embedded_patches = self.projection(patches)  # (batch_size, 49, hidden_dim)
        return embedded_patches


In [17]:
class ClassToken(layers.Layer):
    def __init__(self, hidden_dim):
        super().__init__()
        # A trainable weight of shape (1, 1, hidden_dim)
        self.cls_token = self.add_weight(
            shape=(1, 1, hidden_dim),
            initializer="random_normal",
            trainable=True
        )

    def call(self, x):
        # x shape: (batch_size, num_patches, hidden_dim)
        batch_size = tf.shape(x)[0]

        # Broadcast the class token across the batch: (batch_size, 1, hidden_dim)
        cls_tokens = tf.broadcast_to(self.cls_token, [batch_size, 1, tf.shape(x)[-1]])

        # Prepend the class token to patch embeddings
        return tf.concat([cls_tokens, x], axis=1)  # shape: (batch_size, 1 + num_patches, hidden_dim)


In [18]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, num_patches, hidden_dim):
        super().__init__()
        # Create a trainable position embedding
        self.pos_emb = self.add_weight(
            shape=(1, num_patches, hidden_dim),
            initializer="random_normal",
            trainable=True
        )

    def call(self, x):
        # x shape: (batch_size, num_patches, hidden_dim)
        return x + self.pos_emb


In [19]:
class TransformerEncoder(layers.Layer):
    def __init__(self, hidden_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization()
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=hidden_dim)
        self.norm2 = layers.LayerNormalization()

        self.mlp = keras.Sequential([
            layers.Dense(mlp_dim, activation='gelu'),
            layers.Dense(hidden_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x):
        # x shape: (batch_size, sequence_length, hidden_dim)

        # 1. Multi-Head Self-Attention + Residual
        attn_output = self.attn(x, x)  # self-attention
        x = self.norm1(x + attn_output)

        # 2. Feed-Forward Network + Residual
        mlp_output = self.mlp(x)
        x = self.norm2(x + mlp_output)

        return x


In [20]:
def build_vit(
    image_size=28,
    patch_size=4,
    hidden_dim=64,
    num_heads=2,
    mlp_dim=128,
    num_layers=2,
    num_classes=10
):
    # Number of patches along each dimension
    num_patches_h = image_size // patch_size
    num_patches_w = image_size // patch_size
    total_patches = num_patches_h * num_patches_w

    inputs = keras.Input(shape=(image_size, image_size, 1))  # (28,28,1)

    # 1. Patch Embedding
    x = PatchEmbedding(patch_size, hidden_dim)(inputs)  # (batch_size, 49, hidden_dim)

    # 2. Class Token
    x = ClassToken(hidden_dim)(x)  # (batch_size, 50, hidden_dim)

    # 3. Positional Embedding
    x = PositionalEmbedding(total_patches + 1, hidden_dim)(x)  # (batch_size, 50, hidden_dim)

    # 4. Transformer Encoders
    for _ in range(num_layers):
        x = TransformerEncoder(hidden_dim, num_heads, mlp_dim)(x)

    # 5. Classification Head
    # The [CLS] token's output is at index 0
    cls_token = x[:, 0, :]  # shape: (batch_size, hidden_dim)

    # Final Dense layer for classification
    outputs = layers.Dense(num_classes, activation="softmax")(cls_token)

    # Create the model
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model


In [21]:
# Build the model
vit_model = build_vit()

# Compile
vit_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train
vit_model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=5,
    batch_size=64
)

# Evaluate
test_loss, test_acc = vit_model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")


Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - accuracy: 0.6068 - loss: 1.1158 - val_accuracy: 0.9295 - val_loss: 0.2192
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.9249 - loss: 0.2382 - val_accuracy: 0.9510 - val_loss: 0.1555
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9436 - loss: 0.1746 - val_accuracy: 0.9630 - val_loss: 0.1180
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.9538 - loss: 0.1390 - val_accuracy: 0.9609 - val_loss: 0.1225
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9641 - loss: 0.1147 - val_accuracy: 0.9671 - val_loss: 0.1037
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9631 - loss: 0.1205
Test accuracy: 0.9671
